# Topic Modeling Exploration

This notebook explores whether topic modeling can be used to compare the word distribution / vocabulary of characters.
We try two approaches:

1. **BERTopic** — transformer-based topic modeling that handles short texts well
2. **Word Embedding Clustering** — cluster words by semantic similarity, then compare character usage per cluster


## Setup


In [1]:
import pandas as pd
import numpy as np
import altair as alt

df = pd.read_csv("../data/clean_data/simpsons_script_lines_clean.csv")
alt.data_transformers.enable("vegafusion")
df.head()

,episode_id,season,number_in_season,title,imdb_rating,line_number,character,location_id,location,spoken_words,word_count,sentence_count
0,1,1,1,Simpsons Roasting on an Open Fire,8.2,2,Marge Simpson,2,Car,"Ooo, careful, Homer.",3,1
1,1,1,1,Simpsons Roasting on an Open Fire,8.2,3,Homer Simpson,2,Car,There's no time to be careful.,6,1
2,1,1,1,Simpsons Roasting on an Open Fire,8.2,4,Homer Simpson,2,Car,We're late.,2,1
3,1,1,1,Simpsons Roasting on an Open Fire,8.2,7,Marge Simpson,4,Auditorium,"Sorry, Excuse us. Pardon me...",5,2
4,1,1,1,Simpsons Roasting on an Open Fire,8.2,8,Homer Simpson,4,Auditorium,"Hey, Norman. How's it going? So you got dragge...",21,6


## Approach 1: BERTopic

**Idea:** Automatically discover "topics" (themes/subjects) in the dialogue, then compare which topics each character talks about most. For example, we might discover a "food/beer" topic, a "school" topic, a "money" topic — and then see that Homer dominates the food topic while Lisa dominates the school topic.

**How it works:**

1. We build "documents" by grouping consecutive lines spoken by the same character at the same location in an episode (a "scene segment"). This gives us topically coherent chunks of dialogue.
2. BERTopic embeds each document into a high-dimensional vector using a sentence transformer model (captures semantic meaning, not just keywords).
3. It reduces dimensionality with UMAP, then clusters similar documents together with HDBSCAN.
4. Each cluster becomes a "topic." The topic is labeled by the most distinctive words in that cluster (using c-TF-IDF — like TF-IDF but across clusters instead of documents).
5. We assign each document to a topic, then count how often each character's documents fall into each topic.

**Key concepts:**

- **Topic -1** = the "outlier" bucket. Documents that didn't fit any cluster well. A high count here means many documents are too generic to assign.
- **c-TF-IDF** = class-based TF-IDF. Finds words that are distinctive to a topic compared to all other topics.
- We use `CountVectorizer(stop_words="english")` so that topic labels show meaningful words instead of "you", "the", "it".

**Dependencies:** `pip install bertopic`  
(also installs `sentence-transformers`, `hdbscan`, `umap-learn`)


In [2]:
# Build scene-segment documents using location and line_number from clean data
# A "segment" = consecutive lines by the same character at the same location in the same episode
# A new segment starts when character/location changes or there's a gap > 20 lines

top_characters = df.groupby("character")["word_count"].sum().nlargest(10).index.tolist()
raw_top = (
    df[df["character"].isin(top_characters)]
    .sort_values(["episode_id", "line_number"])
    .reset_index(drop=True)
)

segments = []
current_segment = []
prev_episode = prev_char = prev_loc = prev_num = None

for _, row in raw_top.iterrows():
    same_episode = row["episode_id"] == prev_episode
    same_char = row["character"] == prev_char
    same_loc = row["location_id"] == prev_loc
    close_lines = (
        (row["line_number"] - prev_num <= 20) if prev_num is not None else False
    )

    if same_episode and same_char and same_loc and close_lines:
        current_segment.append(row)
    else:
        if current_segment:
            segments.append(current_segment)
        current_segment = [row]

    prev_episode = row["episode_id"]
    prev_char = row["character"]
    prev_loc = row["location_id"]
    prev_num = row["line_number"]

if current_segment:
    segments.append(current_segment)

# Aggregate each segment into a document
docs_rows = []
for seg in segments:
    text = " ".join(r["spoken_words"] for r in seg)
    docs_rows.append(
        {
            "character": seg[0]["character"],
            "episode_id": seg[0]["episode_id"],
            "season": seg[0]["season"],
            "location_id": seg[0]["location_id"],
            "location": seg[0]["location"],
            "spoken_words": text,
            "n_lines": len(seg),
        }
    )

docs_df = pd.DataFrame(docs_rows)

# Filter out very short segments (< 3 lines) — not enough text for meaningful topics
docs_df = docs_df[docs_df["n_lines"] >= 3].reset_index(drop=True)

print(f"Number of scene-segment documents: {len(docs_df)}")
print(f"Avg lines per segment: {docs_df['n_lines'].mean():.1f}")
print(f"\nTop locations:")
print(docs_df["location"].value_counts().head(10))
docs_df.head()

Number of scene-segment documents: 5382
Avg lines per segment: 4.2

Top locations:
location
Simpson Home                       1009
Springfield Elementary School       289
Springfield Nuclear Power Plant     196
Moe's Tavern                        190
Kwik-E-Mart                          64
Springfield Street                   52
Burns Manor                          49
First Church of Springfield          45
Springfield                          44
Flanders Home                        41
Name: count, dtype: int64


,character,episode_id,season,location_id,location,spoken_words,n_lines
0,Homer Simpson,1,1,5,Simpson Home,"Y'ello. Who's this? This is her sister, isn't ...",5
1,Bart Simpson,1,1,5,Simpson Home,Ow! Quit it. Ow! Quit it! Ow! Quit it! Ow! Qui...,4
2,Homer Simpson,1,1,15,Moe's Tavern,"Thanks, Moe. What's with the crazy getup, Barn...",3
3,Homer Simpson,1,1,16,Santa School,"Huh, when do we get paid? Dasher, Dancer... Pr...",7
4,Homer Simpson,1,1,5,Simpson Home,"What? Why? Oh, yeah. Hello Patty, hello Selma....",5


In [3]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# Use a vectorizer with stop words removed for cleaner topic representations
vectorizer = CountVectorizer(stop_words="english")

# Fit BERTopic on scene-segment documents
topic_model = BERTopic(
    embedding_model="all-MiniLM-L6-v2",  # fast, good quality
    vectorizer_model=vectorizer,  # removes stop words from topic labels
    min_topic_size=10,
    verbose=True,
)

topics, probs = topic_model.fit_transform(docs_df["spoken_words"].tolist())

/Users/I549663/code/uni/dv/data-vis-simpsons/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-13 16:36:54,481 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 169/169 [00:07<00:00, 21.22it/s]
2026-05-13 16:37:07,984 - BERTopic - Embedding - Completed ✓
2026-05-13 16:37:07,984 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-13 16:37:17,892 - BERTopic - Dimensionality - Completed ✓
2026-05-13 16:37:17,896 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-13 16:37:18,047 - BERTopic - Cluster - Completed ✓
2026-05-13 16:37:18,050 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-13 16:37:18,145 - BERTopic - Representation - Completed ✓


In [4]:
# View discovered topics
topic_model.get_topic_info().head(20)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,2584,-1_oh_just_ll_like,"[oh, just, ll, like, don, know, hey, uh, right...","[Hey! I'm Bart Simpson's father, and I'm sick ..."
1,0,242,0_chocolate_cream_eat_oh,"[chocolate, cream, eat, oh, food, ll, ice, lik...","[Hey Apu. Sittin' in the ice cream cooler, eh?..."
2,1,159,1_school_class_chalmers_superintendent,"[school, class, chalmers, superintendent, grad...",[Superintendent Chalmers! Welcome! So what's t...
3,2,148,2_marge_ll_like_oh,"[marge, ll, like, oh, wife, girl, just, okay, ...",[WITH MY... Wait'll I tell Marge! Well... if I...
4,3,126,3_money_dollars_cash_million,"[money, dollars, cash, million, thousand, need...","[Need money. Need money. One Scratch-for-Cash,..."
5,4,103,4_homer_simpson_doink_shoo,"[homer, simpson, doink, shoo, yoy, park, mind,...",[One night. The one night of the year I want H...
6,5,97,5_milhouse_yoko_soul_van,"[milhouse, yoko, soul, van, houten, friends, n...","[Milhouse? Is that you? Uh, maybe later. Milho..."
7,6,85,6_church_reverend_god_bible,"[church, reverend, god, bible, goliath, jesus,...","[Sorry to bother you, Reverend Lovejoy, but I'..."
8,7,85,7_police_skinner_cop_tony,"[police, skinner, cop, tony, chief, fat, offic...","[Boys, even though I've been made Police Commi..."
9,8,84,8_dad_father_daddy_little,"[dad, father, daddy, little, look, old, love, ...",[Here's where it all started to go wrong. How'...


In [5]:
# Generate topic summaries for LLM labeling
# Copy this output and paste it into an LLM to get short 2-3 word topic labels

print("=" * 80)
print("PROMPT: For each topic below, generate a short label (2-3 words max) that")
print("captures the theme. These are topics discovered in Simpsons dialogue.")
print("Format your response as: Topic X: <label>")
print("=" * 80)
print()

topic_info = topic_model.get_topic_info().query("Topic != -1")

for _, row in topic_info.iterrows():
    topic_id = row["Topic"]
    count = row["Count"]
    top_words = row["Representation"][:10]

    # Get a short representative doc snippet
    rep_docs = row.get("Representative_Docs", [])
    snippet = rep_docs[0][:150] + "..." if rep_docs and len(rep_docs) > 0 else "N/A"

    print(f"--- Topic {topic_id} ({count} documents) ---")
    print(f"Top words: {', '.join(top_words)}")
    print(f"Example: {snippet}")
    print()

PROMPT: For each topic below, generate a short label (2-3 words max) that
captures the theme. These are topics discovered in Simpsons dialogue.
Format your response as: Topic X: <label>

--- Topic 0 (242 documents) ---
Top words: chocolate, cream, eat, oh, food, ll, ice, like, pie, chicken
Example: Hey Apu. Sittin' in the ice cream cooler, eh? Whoa, too much information! Thanks for the mental picture. Why don't you tell us what you really think? ...

--- Topic 1 (159 documents) ---
Top words: school, class, chalmers, superintendent, grade, children, hoover, principal, students, professor
Example: Superintendent Chalmers! Welcome! So what's the word down at One School Board Plaza? Very good. Back to the three R's. Hm. What do you think of the ba...

--- Topic 2 (148 documents) ---
Top words: marge, ll, like, oh, wife, girl, just, okay, gotta, don
Example: WITH MY... Wait'll I tell Marge! Well... if I explain it to Marge that way, I'm sure she'll understand....

--- Topic 3 (126 document

In [6]:
# Paste the LLM response here between the triple quotes, then run this cell.
# Expected format — one line per topic: "Topic X: <label>"

llm_response = """
Topic 0: Food & Snacks
Topic 1: Marge & Homer
Topic 2: Movies & TV
Topic 3: School Administration
Topic 4: Homer's Catchphrases
Topic 5: Milhouse Moments
Topic 6: Money & Wealth
Topic 7: Church & Religion
Topic 8: Driving & Cars
Topic 9: Maggie Baby Talk
Topic 10: Bart Simpson
Topic 11: Father-Son Bonds
Topic 12: Christmas & Pets
Topic 13: Springfield Town
Topic 14: Ned Flanders
Topic 15: Animals
Topic 16: Police
Topic 17: Politics & Elections
Topic 18: Smithers & Burns
Topic 19: Nuclear Plant
Topic 20: Numbers & Counting
Topic 21: Doctors & Medical
Topic 22: Valentine's & Weddings
Topic 23: Death & Dying
Topic 24: Mr. Burns
Topic 25: Hair & Grooming
Topic 26: Exclamations
Topic 27: Clothing & Style
Topic 28: Krusty the Clown
Topic 29: Moe's Tavern
Topic 30: Lisa Simpson
Topic 31: Baseball
Topic 32: Football & Coach
Topic 33: Phones
Topic 34: Doors & Buttons
Topic 35: Marriage Troubles
Topic 36: Guns & Shooting
Topic 37: Dating & Romance
Topic 38: Jobs & Employment
Topic 39: Friendship
Topic 40: Dance & Ballet
Topic 41: Nelson the Bully
Topic 42: Lenny & Carl
Topic 43: Motherhood
Topic 44: Letters & Heroes
Topic 45: Farm Animals
Topic 46: Glasses & Vision
Topic 47: Apu & Kwik-E-Mart
Topic 48: Days & Nights
Topic 49: Violence & Revenge
Topic 50: Poe Parody
Topic 51: Principal Skinner
Topic 52: America & Geography
Topic 53: Electricity & Inventions
Topic 54: Firing & Unions
Topic 55: Family Bonds
Topic 56: Grampa Simpson
Topic 57: Officer Lou
Topic 58: Mrs. Krabappel
Topic 59: Stock Market
Topic 60: Grampa's Advice
Topic 61: House & Home
Topic 62: Bart Strangling
Topic 63: Tickets & Prizes
Topic 64: Merchandise
Topic 65: Sleep & Beds
Topic 66: Marriage & Affairs
Topic 67: Jessica Lovejoy
Topic 68: Gay Themes
"""

# Parse the response into a dict
import re

custom_topic_labels = {}
for line in llm_response.strip().split("\n"):
    match = re.match(r"Topic\s+(\d+):\s*(.+)", line.strip())
    if match:
        topic_id = int(match.group(1))
        label = match.group(2).strip()
        custom_topic_labels[topic_id] = label

print(f"Parsed {len(custom_topic_labels)} topic labels:")
for tid, label in sorted(custom_topic_labels.items()):
    print(f"  Topic {tid}: {label}")

# Store as topic_names_custom — will be merged with BERTopic names later in the chart cells
# (fallback to BERTopic default names happens there, not here)
topic_names_custom = custom_topic_labels.copy()

Parsed 69 topic labels:
  Topic 0: Food & Snacks
  Topic 1: Marge & Homer
  Topic 2: Movies & TV
  Topic 3: School Administration
  Topic 4: Homer's Catchphrases
  Topic 5: Milhouse Moments
  Topic 6: Money & Wealth
  Topic 7: Church & Religion
  Topic 8: Driving & Cars
  Topic 9: Maggie Baby Talk
  Topic 10: Bart Simpson
  Topic 11: Father-Son Bonds
  Topic 12: Christmas & Pets
  Topic 13: Springfield Town
  Topic 14: Ned Flanders
  Topic 15: Animals
  Topic 16: Police
  Topic 17: Politics & Elections
  Topic 18: Smithers & Burns
  Topic 19: Nuclear Plant
  Topic 20: Numbers & Counting
  Topic 21: Doctors & Medical
  Topic 22: Valentine's & Weddings
  Topic 23: Death & Dying
  Topic 24: Mr. Burns
  Topic 25: Hair & Grooming
  Topic 26: Exclamations
  Topic 27: Clothing & Style
  Topic 28: Krusty the Clown
  Topic 29: Moe's Tavern
  Topic 30: Lisa Simpson
  Topic 31: Baseball
  Topic 32: Football & Coach
  Topic 33: Phones
  Topic 34: Doors & Buttons
  Topic 35: Marriage Troubles
  Top

In [7]:
# Assign topics back to the documents
docs_df = docs_df.assign(topic=topics)

# Topic name lookup (original BERTopic names as fallback)
topic_names = {
    row["Topic"]: row["Name"] for _, row in topic_model.get_topic_info().iterrows()
}

# Compare topic distribution between two characters
character_a = "Homer Simpson"
character_b = "Marge Simpson"
n_topics = 15  # change this to show more/fewer topics
use_custom_labels = True  # toggle: True = LLM labels, False = BERTopic default names

top_topics = (
    topic_model.get_topic_info()
    .query("Topic != -1")
    .nlargest(n_topics, "Count")["Topic"]
    .tolist()
)

topic_dist = (
    docs_df[
        (docs_df["character"].isin([character_a, character_b]))
        & (docs_df["topic"].isin(top_topics))
    ]
    .groupby(["character", "topic"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

# Normalize to proportions per character
char_totals = topic_dist.groupby("character")["count"].sum()
topic_dist = topic_dist.assign(
    proportion=topic_dist.apply(
        lambda r: r["count"] / char_totals[r["character"]], axis=1
    )
)

labels_to_use = (
    topic_names_custom
    if (use_custom_labels and "topic_names_custom" in dir())
    else topic_names
)
topic_dist = topic_dist.assign(topic_label=topic_dist["topic"].map(labels_to_use))

alt.Chart(topic_dist).mark_bar().encode(
    x=alt.X(
        "topic_label:N", title="Topic", sort=top_topics, axis=alt.Axis(labelAngle=-45)
    ),
    y=alt.Y("proportion:Q", title="Proportion of Documents"),
    color="character:N",
    xOffset="character:N",
    tooltip=["character", "topic_label", alt.Tooltip("proportion:Q", format=".2%")],
).properties(
    title=f"Topic Distribution (Top {n_topics}): {character_a} vs {character_b}",
    width=700,
    height=400,
)

alt.Chart(...)

In [8]:
# Topic distribution for the 4 main characters
main_characters = ["Homer Simpson", "Marge Simpson", "Bart Simpson", "Lisa Simpson"]
n_topics = 15  # change this to show more/fewer topics
use_custom_labels = False  # toggle: True = LLM labels, False = BERTopic default names

top_topics_4 = (
    topic_model.get_topic_info()
    .query("Topic != -1")
    .nlargest(n_topics, "Count")["Topic"]
    .tolist()
)

topic_dist_4 = (
    docs_df[
        (docs_df["character"].isin(main_characters))
        & (docs_df["topic"].isin(top_topics_4))
    ]
    .groupby(["character", "topic"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

# Normalize to proportions per character
char_totals_4 = topic_dist_4.groupby("character")["count"].sum()
topic_dist_4 = topic_dist_4.assign(
    proportion=topic_dist_4.apply(
        lambda r: r["count"] / char_totals_4[r["character"]], axis=1
    )
)

labels_to_use = (
    topic_names_custom
    if (use_custom_labels and "topic_names_custom" in dir())
    else topic_names
)
topic_dist_4 = topic_dist_4.assign(topic_label=topic_dist_4["topic"].map(labels_to_use))

alt.Chart(topic_dist_4).mark_bar().encode(
    x=alt.X(
        "topic_label:N", title="Topic", sort=top_topics_4, axis=alt.Axis(labelAngle=-45)
    ),
    y=alt.Y("proportion:Q", title="Proportion of Documents"),
    color="character:N",
    xOffset="character:N",
    tooltip=["character", "topic_label", alt.Tooltip("proportion:Q", format=".2%")],
).properties(
    title=f"Topic Distribution (Top {n_topics}): Simpson Family",
    width=800,
    height=400,
)

alt.Chart(...)

---

## Approach 2: Word Embedding Clustering

**Idea:** Instead of discovering topics from full sentences (like BERTopic), we work at the individual word level. We embed each word into a vector space where semantically similar words are close together, then cluster them into groups. These clusters become "pseudo-topics" (e.g. a cluster of food words, a cluster of family words). We then compare how much each character uses words from each cluster.

**How it works:**

1. **Tokenize & filter:** Extract all words used by top characters. Keep only words that appear ≥20 times (reliable signal) AND have a TF-IDF score above median (distinctive to at least one character). This removes both rare noise and generic filler.
2. **Embed:** Look up each word's pre-trained vector from GloVe (trained on billions of words of text). Each word becomes a 100-dimensional point in space where "chicken" is near "bacon" and "school" is near "teacher."
3. **Cluster:** Use KMeans to group nearby word vectors into k=15 clusters. Each cluster represents a semantic theme.
4. **Compare:** For each character, count how often they use words from each cluster. Normalize to proportions so we can compare across characters who speak different total amounts.
5. **Visualize:** Project the high-dimensional word vectors to 2D (using UMAP or t-SNE) to create a visual "map" of the vocabulary, then use dot size/color to show character differences.

**Key difference from BERTopic:** BERTopic works on sentence-level meaning (context matters — "bank" near "river" vs "bank" near "money" are different). This approach works on word-level meaning (each word has one fixed position regardless of context). Simpler but more interpretable — you can see exactly which words drive the differences.

**Dependencies:** `pip install gensim` (for pre-trained word vectors)


In [9]:
import re
from collections import Counter

import nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))


def tokenize(text):
    return [
        w
        for w in re.findall(r"[a-z]+", text.lower())
        if w not in stop_words and len(w) > 2
    ]


# Build vocabulary from top characters
top_characters = df.groupby("character")["word_count"].sum().nlargest(10).index.tolist()
top_df = df[df["character"].isin(top_characters)]

# Count word frequencies across all top characters
all_tokens = []
for text in top_df["spoken_words"]:
    all_tokens.extend(tokenize(text))

vocab_counts = Counter(all_tokens)

# --- Filtering Step 1: minimum frequency ---
min_freq = 20
freq_vocab = {word: count for word, count in vocab_counts.items() if count >= min_freq}
print(f"Words with freq >= {min_freq}: {len(freq_vocab)}")

# --- Filtering Step 2: compute TF-IDF and keep top 50% ---
# Treat each character as a "document"
char_word_counts = {}
for character in top_characters:
    char_lines = top_df[top_df["character"] == character]["spoken_words"]
    char_tokens = []
    for text in char_lines:
        char_tokens.extend(tokenize(text))
    char_word_counts[character] = Counter(char_tokens)

# IDF: log(num_characters / num_characters_using_word)
n_chars = len(top_characters)
word_idf = {}
for word in freq_vocab:
    doc_freq = sum(1 for c in top_characters if char_word_counts[c].get(word, 0) > 0)
    word_idf[word] = np.log(n_chars / doc_freq) if doc_freq > 0 else 0

# Max TF-IDF across characters (how distinctive is this word for any character?)
word_max_tfidf = {}
for word in freq_vocab:
    max_tfidf = 0
    for character in top_characters:
        total = sum(char_word_counts[character].values())
        tf = char_word_counts[character].get(word, 0) / total if total > 0 else 0
        tfidf = tf * word_idf[word]
        max_tfidf = max(max_tfidf, tfidf)
    word_max_tfidf[word] = max_tfidf

# Keep words with TF-IDF above median (top 50%)
tfidf_threshold = np.percentile(list(word_max_tfidf.values()), 50)
vocab = [w for w in freq_vocab if word_max_tfidf[w] >= tfidf_threshold]

print(
    f"Words after TF-IDF filter (top 50%, threshold={tfidf_threshold:.6f}): {len(vocab)}"
)
print(f"Filtered out {len(freq_vocab) - len(vocab)} generic words")

Words with freq >= 20: 2404
Words after TF-IDF filter (top 50%, threshold=0.000050): 1205
Filtered out 1199 generic words


In [10]:
import gensim.downloader as api

# Load pre-trained word vectors (smaller model for speed)
# Options: "glove-wiki-gigaword-100", "word2vec-google-news-300"
print("Loading word vectors (this may take a minute)...")
wv = api.load("glove-wiki-gigaword-100")
print("Done.")

# Get embeddings only for our filtered vocabulary
words_with_vectors = [w for w in vocab if w in wv]
word_vectors = np.array([wv[w] for w in words_with_vectors])

print(f"Words with embeddings: {len(words_with_vectors)} / {len(vocab)}")
print(
    f"(Started with {len(vocab_counts)} unique tokens → filtered to {len(words_with_vectors)} for clustering)"
)

Loading word vectors (this may take a minute)...
Done.
Words with embeddings: 1199 / 1205
(Started with 27459 unique tokens → filtered to 1199 for clustering)


In [11]:
from sklearn.cluster import KMeans

# Get embeddings for words in our vocabulary that exist in the pre-trained model
words_with_vectors = [w for w in vocab if w in wv]
word_vectors = np.array([wv[w] for w in words_with_vectors])

print(f"Words with embeddings: {len(words_with_vectors)} / {len(vocab)}")

# Cluster into k semantic groups
k = 15
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(word_vectors)

# Create word-to-cluster mapping
word_cluster = pd.DataFrame({"word": words_with_vectors, "cluster": cluster_labels})

# Show representative words per cluster (closest to centroid)
for i in range(k):
    cluster_words = word_cluster[word_cluster["cluster"] == i]["word"].tolist()
    # Sort by distance to centroid
    dists = [np.linalg.norm(wv[w] - kmeans.cluster_centers_[i]) for w in cluster_words]
    sorted_words = [w for _, w in sorted(zip(dists, cluster_words))][:8]
    print(f"Cluster {i:2d}: {', '.join(sorted_words)}")

Words with embeddings: 1199 / 1205
Cluster  0: starting, winning, losing, opening, third, pair, contest, round
Cluster  1: although, however, decided, soon, later, immediately, leaving, following
Cluster  2: awww, aww, nooo, eww, ahhh, ewww, uhh, nooooo
Cluster  3: instead, bringing, putting, besides, giving, certain, possibly, beyond
Cluster  4: indeed, unfortunately, surely, imagine, sort, certainly, hardly, realize
Cluster  5: picked, grab, hook, broken, rush, stopped, onto, trap
Cluster  6: presents, created, original, reality, popular, hollywood, stage, cast
Cluster  7: cat, ape, ghost, rat, pig, monkeys, pet, shark
Cluster  8: sister, daughter, husband, girlfriend, aunt, boyfriend, mom, sisters
Cluster  9: toast, snack, sandwiches, chicken, pie, bread, fried, peanut
Cluster 10: buck, becky, maggie, valentine, janey, nelson, jimmy, eddie
Cluster 11: teaching, learning, taught, student, teacher, teach, teachers, education
Cluster 12: others, responsible, child, lives, criminal, dan

In [12]:
# Count how often each character uses words from each cluster
word_to_cluster = dict(zip(word_cluster["word"], word_cluster["cluster"]))

cluster_usage_rows = []
for character in top_characters:
    char_lines = top_df[top_df["character"] == character]["spoken_words"]
    char_tokens = []
    for text in char_lines:
        char_tokens.extend(tokenize(text))

    # Count tokens per cluster
    cluster_counts = Counter()
    for token in char_tokens:
        if token in word_to_cluster:
            cluster_counts[word_to_cluster[token]] += 1

    total = sum(cluster_counts.values())
    for cluster_id, count in cluster_counts.items():
        cluster_usage_rows.append(
            {
                "character": character,
                "cluster": cluster_id,
                "count": count,
                "proportion": count / total if total > 0 else 0,
            }
        )

cluster_usage = pd.DataFrame(cluster_usage_rows)
cluster_usage.head()

,character,cluster,count,proportion
0,Homer Simpson,4,2520,0.117030
1,Homer Simpson,11,507,0.023545
2,Homer Simpson,13,2107,0.097850
3,Homer Simpson,2,2119,0.098407
4,Homer Simpson,8,496,0.023034


In [13]:
# Compare cluster usage between two characters
character_a = "Homer Simpson"
character_b = "Marge Simpson"

comparison = cluster_usage[cluster_usage["character"].isin([character_a, character_b])]

# Build readable cluster labels from top 3 representative words
cluster_label_map = {}
cluster_order = []
for cluster_id in range(k):
    cluster_words = word_cluster.loc[
        word_cluster["cluster"] == cluster_id, "word"
    ].tolist()
    distances = [
        np.linalg.norm(wv[w] - kmeans.cluster_centers_[cluster_id])
        for w in cluster_words
    ]
    top_words = [w for _, w in sorted(zip(distances, cluster_words))[:3]]
    label = f"{', '.join(top_words)}"
    cluster_label_map[cluster_id] = label
    cluster_order.append(label)

comparison = comparison.copy()
comparison["cluster_label"] = comparison["cluster"].map(cluster_label_map)

alt.Chart(comparison).mark_bar().encode(
    x=alt.X(
        "cluster_label:N",
        title="Semantic Cluster",
        sort=cluster_order,
        axis=alt.Axis(labelAngle=-45),
    ),
    y=alt.Y("proportion:Q", title="Proportion of Words", axis=alt.Axis(format="%")),
    color="character:N",
    xOffset="character:N",
    tooltip=["character", "cluster_label", alt.Tooltip("proportion:Q", format=".2%")],
).properties(
    title=f"Semantic Cluster Usage: {character_a} vs {character_b}",
    width=700,
    height=400,
)

alt.Chart(...)

In [14]:
# Compare cluster usage — 4 main characters
main_characters = ["Homer Simpson", "Marge Simpson", "Bart Simpson", "Lisa Simpson"]

comparison_4 = cluster_usage[cluster_usage["character"].isin(main_characters)].copy()
comparison_4["cluster_label"] = comparison_4["cluster"].map(cluster_label_map)

alt.Chart(comparison_4).mark_bar().encode(
    x=alt.X(
        "cluster_label:N",
        title="Semantic Cluster",
        sort=cluster_order,
        axis=alt.Axis(labelAngle=-45),
    ),
    y=alt.Y("proportion:Q", title="Proportion of Words", axis=alt.Axis(format="%")),
    color="character:N",
    xOffset="character:N",
    tooltip=["character", "cluster_label", alt.Tooltip("proportion:Q", format=".2%")],
).properties(
    title="Semantic Cluster Usage: Simpson Family",
    width=800,
    height=400,
)

alt.Chart(...)

In [15]:
# 2D scatter plot of word embeddings colored by cluster (UMAP projection)
from umap import UMAP

# Reduce high-dimensional word vectors to 2D for visualization
reducer = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
coords_2d = reducer.fit_transform(word_vectors)

# Add 2D coordinates to word_cluster DataFrame
word_cluster = word_cluster.assign(x=coords_2d[:, 0], y=coords_2d[:, 1])

# Compute axis domains trimmed to 1st/99th percentile (avoids outliers stretching the view)
x_lo, x_hi = word_cluster["x"].quantile(0.01), word_cluster["x"].quantile(0.99)
y_lo, y_hi = word_cluster["y"].quantile(0.01), word_cluster["y"].quantile(0.99)
padding = 0.5
x_domain = [x_lo - padding, x_hi + padding]
y_domain = [y_lo - padding, y_hi + padding]

# Interactive scatter plot — each dot is a word, colored by cluster
alt.Chart(word_cluster).mark_circle(size=40, opacity=0.7).encode(
    x=alt.X(
        "x:Q",
        title="UMAP 1",
        axis=alt.Axis(labels=False, ticks=False),
        scale=alt.Scale(domain=x_domain),
    ),
    y=alt.Y(
        "y:Q",
        title="UMAP 2",
        axis=alt.Axis(labels=False, ticks=False),
        scale=alt.Scale(domain=y_domain),
    ),
    color=alt.Color("cluster:N", title="Cluster", scale=alt.Scale(scheme="category20")),
    tooltip=["word", "cluster"],
).properties(
    title="Word Embedding Space (UMAP projection, colored by KMeans cluster)",
    width=600,
    height=500,
)

alt.Chart(...)

In [16]:
# 2D scatter plot — t-SNE projection for comparison with UMAP
from sklearn.manifold import TSNE

# Reduce to 2D using t-SNE (preserves local structure, produces tighter clusters)
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
coords_tsne = tsne.fit_transform(word_vectors)

word_cluster_tsne = word_cluster[["word", "cluster"]].copy()
word_cluster_tsne["x"] = coords_tsne[:, 0]
word_cluster_tsne["y"] = coords_tsne[:, 1]

# Trimmed axes
x_lo_t, x_hi_t = (
    word_cluster_tsne["x"].quantile(0.01),
    word_cluster_tsne["x"].quantile(0.99),
)
y_lo_t, y_hi_t = (
    word_cluster_tsne["y"].quantile(0.01),
    word_cluster_tsne["y"].quantile(0.99),
)
padding_t = 2
x_domain_tsne = [x_lo_t - padding_t, x_hi_t + padding_t]
y_domain_tsne = [y_lo_t - padding_t, y_hi_t + padding_t]

alt.Chart(word_cluster_tsne).mark_circle(size=40, opacity=0.7).encode(
    x=alt.X(
        "x:Q",
        title="t-SNE 1",
        axis=alt.Axis(labels=False, ticks=False),
        scale=alt.Scale(domain=x_domain_tsne),
    ),
    y=alt.Y(
        "y:Q",
        title="t-SNE 2",
        axis=alt.Axis(labels=False, ticks=False),
        scale=alt.Scale(domain=y_domain_tsne),
    ),
    color=alt.Color("cluster:N", title="Cluster", scale=alt.Scale(scheme="category20")),
    tooltip=["word", "cluster"],
).properties(
    title="Word Embedding Space (t-SNE projection, colored by KMeans cluster)",
    width=600,
    height=500,
)

alt.Chart(...)

In [17]:
# 2D scatter per character — dot size = how much that character uses the word
character_a = "Homer Simpson"
character_b = "Marge Simpson"


# Get word usage counts per character (normalized to proportions)
def get_char_word_proportions(character):
    char_lines = top_df[top_df["character"] == character]["spoken_words"]
    tokens = []
    for text in char_lines:
        tokens.extend(tokenize(text))
    counts = Counter(tokens)
    total = sum(counts.values())
    return {w: counts.get(w, 0) / total for w in words_with_vectors}


props_a = get_char_word_proportions(character_a)
props_b = get_char_word_proportions(character_b)

# Add character usage to the word_cluster DataFrame
scatter_a = word_cluster.assign(
    usage=word_cluster["word"].map(props_a), character=character_a
)
scatter_b = word_cluster.assign(
    usage=word_cluster["word"].map(props_b), character=character_b
)


def make_char_scatter(data, character):
    return (
        alt.Chart(data)
        .mark_circle(opacity=0.7)
        .encode(
            x=alt.X(
                "x:Q",
                title="UMAP 1",
                axis=alt.Axis(labels=False, ticks=False),
                scale=alt.Scale(domain=x_domain),
            ),
            y=alt.Y(
                "y:Q",
                title="UMAP 2",
                axis=alt.Axis(labels=False, ticks=False),
                scale=alt.Scale(domain=y_domain),
            ),
            size=alt.Size(
                "usage:Q", title="Usage (proportion)", scale=alt.Scale(range=[10, 300])
            ),
            color=alt.Color(
                "cluster:N", title="Cluster", scale=alt.Scale(scheme="category20")
            ),
            tooltip=["word", "cluster", alt.Tooltip("usage:Q", format=".4f")],
        )
        .properties(
            title=character,
            width=400,
            height=400,
        )
    )


chart_a = make_char_scatter(scatter_a, character_a)
chart_b = make_char_scatter(scatter_b, character_b)

(chart_a | chart_b).properties(
    title=alt.TitleParams(
        text="Word Embedding Space — Usage by Character",
        subtitle="Dot size = proportion of character's speech; color = semantic cluster",
        anchor="middle",
    )
)

alt.HConcatChart(...)

In [18]:
# 2D scatter per character — with topic keyword labels at cluster centroids
character_a = "Homer Simpson"
character_b = "Marge Simpson"

# Cluster centroid labels (top 3 representative words)
label_rows = []
for cluster_id in range(k):
    cluster_words = word_cluster[word_cluster["cluster"] == cluster_id]["word"].tolist()
    cx = word_cluster[word_cluster["cluster"] == cluster_id]["x"].mean()
    cy = word_cluster[word_cluster["cluster"] == cluster_id]["y"].mean()
    dists = [
        np.linalg.norm(wv[w] - kmeans.cluster_centers_[cluster_id])
        for w in cluster_words
    ]
    top_words = [w for _, w in sorted(zip(dists, cluster_words))][:3]
    label_rows.append(
        {"cluster": cluster_id, "x": cx, "y": cy, "label": ", ".join(top_words)}
    )

label_df = pd.DataFrame(label_rows)


def make_char_scatter_labeled(data, character):
    dots = (
        alt.Chart(data)
        .mark_circle(opacity=0.7)
        .encode(
            x=alt.X(
                "x:Q",
                title="UMAP 1",
                axis=alt.Axis(labels=False, ticks=False),
                scale=alt.Scale(domain=x_domain),
            ),
            y=alt.Y(
                "y:Q",
                title="UMAP 2",
                axis=alt.Axis(labels=False, ticks=False),
                scale=alt.Scale(domain=y_domain),
            ),
            size=alt.Size("usage:Q", title="Usage", scale=alt.Scale(range=[10, 300])),
            color=alt.Color(
                "cluster:N", title="Cluster", scale=alt.Scale(scheme="category20")
            ),
            tooltip=["word", "cluster", alt.Tooltip("usage:Q", format=".4f")],
        )
    )

    labels = (
        alt.Chart(label_df)
        .mark_text(fontSize=9, fontWeight="bold", opacity=0.8, dy=-10)
        .encode(
            x=alt.X("x:Q", scale=alt.Scale(domain=x_domain)),
            y=alt.Y("y:Q", scale=alt.Scale(domain=y_domain)),
            text="label:N",
        )
    )

    return (dots + labels).properties(title=character, width=400, height=400)


chart_a = make_char_scatter_labeled(scatter_a, character_a)
chart_b = make_char_scatter_labeled(scatter_b, character_b)

(chart_a | chart_b).properties(
    title=alt.TitleParams(
        text="Word Embedding Space — Usage by Character",
        subtitle="Dot size = usage proportion; color = semantic cluster; labels = topic keywords",
        anchor="middle",
    )
)

alt.HConcatChart(...)

In [19]:
# Option 2: Only show each character's top N most-used words
character_a = "Homer Simpson"
character_b = "Marge Simpson"
n_top_words = 150  # only show the N most-used words per character

props_a = get_char_word_proportions(character_a)
props_b = get_char_word_proportions(character_b)

# Get each character's top words by usage
top_words_a = sorted(props_a, key=props_a.get, reverse=True)[:n_top_words]
top_words_b = sorted(props_b, key=props_b.get, reverse=True)[:n_top_words]

# Filter scatter data to only include that character's top words
scatter_a_filtered = word_cluster[word_cluster["word"].isin(top_words_a)].assign(
    usage=lambda d: d["word"].map(props_a)
)
scatter_b_filtered = word_cluster[word_cluster["word"].isin(top_words_b)].assign(
    usage=lambda d: d["word"].map(props_b)
)


def make_filtered_scatter(data, character):
    return (
        alt.Chart(data)
        .mark_circle(opacity=0.7)
        .encode(
            x=alt.X(
                "x:Q",
                title="UMAP 1",
                axis=alt.Axis(labels=False, ticks=False),
                scale=alt.Scale(domain=x_domain),
            ),
            y=alt.Y(
                "y:Q",
                title="UMAP 2",
                axis=alt.Axis(labels=False, ticks=False),
                scale=alt.Scale(domain=y_domain),
            ),
            size=alt.Size("usage:Q", title="Usage", scale=alt.Scale(range=[20, 300])),
            color=alt.Color(
                "cluster:N", title="Cluster", scale=alt.Scale(scheme="category20")
            ),
            tooltip=["word", "cluster", alt.Tooltip("usage:Q", format=".4f")],
        )
        .properties(title=character, width=400, height=400)
    )


chart_a = make_filtered_scatter(scatter_a_filtered, character_a)
chart_b = make_filtered_scatter(scatter_b_filtered, character_b)

(chart_a | chart_b).properties(
    title=alt.TitleParams(
        text=f"Top {n_top_words} Words per Character in Embedding Space",
        subtitle="Only each character's most-used words are shown; gaps reveal vocabulary differences",
        anchor="middle",
    )
)

alt.HConcatChart(...)

In [20]:
# Option 3: Single plot colored by character dominance
# Color encodes which character uses the word more; saturation = how big the difference
character_a = "Homer Simpson"
character_b = "Marge Simpson"

props_a = get_char_word_proportions(character_a)
props_b = get_char_word_proportions(character_b)

# Compute dominance score: positive = character_a uses more, negative = character_b uses more
# Normalized to [-1, 1] range
dominance_data = word_cluster.copy()
dominance_data["usage_a"] = dominance_data["word"].map(props_a)
dominance_data["usage_b"] = dominance_data["word"].map(props_b)
dominance_data["dominance"] = dominance_data.apply(
    lambda r: (r["usage_a"] - r["usage_b"]) / (r["usage_a"] + r["usage_b"])
    if (r["usage_a"] + r["usage_b"]) > 0
    else 0,
    axis=1,
)
# Total usage for size (more used words = bigger dots)
dominance_data["total_usage"] = dominance_data["usage_a"] + dominance_data["usage_b"]

alt.Chart(dominance_data).mark_circle(opacity=0.7).encode(
    x=alt.X(
        "x:Q",
        title="UMAP 1",
        axis=alt.Axis(labels=False, ticks=False),
        scale=alt.Scale(domain=x_domain),
    ),
    y=alt.Y(
        "y:Q",
        title="UMAP 2",
        axis=alt.Axis(labels=False, ticks=False),
        scale=alt.Scale(domain=y_domain),
    ),
    color=alt.Color(
        "dominance:Q",
        title=f"← {character_b}  |  {character_a} →",
        scale=alt.Scale(scheme="redblue", domain=[-1, 1], domainMid=0),
    ),
    size=alt.Size(
        "total_usage:Q", title="Total Usage", scale=alt.Scale(range=[15, 250])
    ),
    tooltip=[
        "word",
        "cluster",
        alt.Tooltip("dominance:Q", format=".2f"),
        alt.Tooltip("usage_a:Q", format=".4f", title=character_a),
        alt.Tooltip("usage_b:Q", format=".4f", title=character_b),
    ],
).properties(
    title=alt.TitleParams(
        text=f"Character Vocabulary Territory — {character_a} vs {character_b}",
        subtitle="Red = Homer's words, Blue = Marge's words, Grey = shared equally",
    ),
    width=600,
    height=500,
)

alt.Chart(...)

In [21]:
# Option 3b: Territory map with cluster boundaries and topic labels
from scipy.spatial import ConvexHull

character_a = "Homer Simpson"
character_b = "Marge Simpson"

# Compute convex hull polygons for each cluster
hull_rows = []
for cluster_id in range(k):
    cluster_points = dominance_data[dominance_data["cluster"] == cluster_id][
        ["x", "y"]
    ].values
    if len(cluster_points) >= 3:
        try:
            hull = ConvexHull(cluster_points)
            vertices = hull.vertices.tolist() + [hull.vertices[0]]
            for order, v_idx in enumerate(vertices):
                hull_rows.append(
                    {
                        "cluster": cluster_id,
                        "x": cluster_points[v_idx, 0],
                        "y": cluster_points[v_idx, 1],
                        "order": order,
                    }
                )
        except Exception:
            pass

hull_df = pd.DataFrame(hull_rows)

# Compute cluster centroids and representative keywords for labels
label_rows = []
for cluster_id in range(k):
    cluster_words = word_cluster[word_cluster["cluster"] == cluster_id]["word"].tolist()
    cx = word_cluster[word_cluster["cluster"] == cluster_id]["x"].mean()
    cy = word_cluster[word_cluster["cluster"] == cluster_id]["y"].mean()
    dists = [
        np.linalg.norm(wv[w] - kmeans.cluster_centers_[cluster_id])
        for w in cluster_words
    ]
    top_words = [w for _, w in sorted(zip(dists, cluster_words))][:3]
    label_rows.append(
        {"cluster": cluster_id, "x": cx, "y": cy, "label": ", ".join(top_words)}
    )

label_df = pd.DataFrame(label_rows)

# Layer 1: Cluster hull outlines (background)
hulls = (
    alt.Chart(hull_df)
    .mark_line(strokeWidth=1.5, opacity=0.4)
    .encode(
        x=alt.X("x:Q", scale=alt.Scale(domain=x_domain)),
        y=alt.Y("y:Q", scale=alt.Scale(domain=y_domain)),
        detail="cluster:N",
        color=alt.Color("cluster:N", scale=alt.Scale(scheme="category20"), legend=None),
        order="order:O",
    )
)

# Layer 2: Dots colored by character dominance
dots = (
    alt.Chart(dominance_data)
    .mark_circle(opacity=0.7)
    .encode(
        x=alt.X(
            "x:Q",
            title="UMAP 1",
            axis=alt.Axis(labels=False, ticks=False),
            scale=alt.Scale(domain=x_domain),
        ),
        y=alt.Y(
            "y:Q",
            title="UMAP 2",
            axis=alt.Axis(labels=False, ticks=False),
            scale=alt.Scale(domain=y_domain),
        ),
        color=alt.Color(
            "dominance:Q",
            title=f"← {character_b}  |  {character_a} →",
            scale=alt.Scale(scheme="redblue", domain=[-1, 1], domainMid=0),
        ),
        size=alt.Size(
            "total_usage:Q", title="Total Usage", scale=alt.Scale(range=[15, 250])
        ),
        tooltip=[
            "word",
            "cluster",
            alt.Tooltip("dominance:Q", format=".2f"),
            alt.Tooltip("usage_a:Q", format=".4f", title=character_a),
            alt.Tooltip("usage_b:Q", format=".4f", title=character_b),
        ],
    )
)

# Layer 3: Cluster keyword labels at centroids
labels = (
    alt.Chart(label_df)
    .mark_text(fontSize=10, fontWeight="bold", opacity=0.8, dy=-12)
    .encode(
        x=alt.X("x:Q", scale=alt.Scale(domain=x_domain)),
        y=alt.Y("y:Q", scale=alt.Scale(domain=y_domain)),
        text="label:N",
    )
)

(hulls + dots + labels).properties(
    title=alt.TitleParams(
        text=f"Character Vocabulary Territory — {character_a} vs {character_b}",
        subtitle="Red = Homer's words, Blue = Marge's words | Outlines = cluster boundaries | Labels = topic keywords",
    ),
    width=650,
    height=550,
)

alt.LayerChart(...)

In [22]:
# Heatmap: all top characters x clusters
alt.Chart(cluster_usage).mark_rect().encode(
    x=alt.X("cluster:N", title="Semantic Cluster"),
    y=alt.Y("character:N", title="Character"),
    color=alt.Color(
        "proportion:Q", title="Proportion", scale=alt.Scale(scheme="blues")
    ),
    tooltip=["character", "cluster", alt.Tooltip("proportion:Q", format=".2%")],
).properties(
    title="Semantic Cluster Usage Across Top Characters",
    width=500,
    height=300,
)

alt.Chart(...)

## Observations

_TODO: Note which approach gave more useful/interpretable results and whether either could be integrated into the main visualization._
